In [ ]:
import numpy as np
from scipy.integrate import odeint
from scipy.optimize import minimize

# --- 1. ENTER YOUR SURVEY DATA HERE ---
# N is your total population size
N = 1000 

# The time points of your surveys (e.g., weeks 0, 1, 2, 3 for your 4 phases)
t_data = np.array([0, 1, 2, 3]) 

# The counts from your surveys at Phase 1, 2, 3, and 4
# Notice Phase 1 (index 0) has 0 Recovered, matching your setup
S_data = np.array([990, 850, 600, 300]) # Example Susceptible counts
I_data = np.array([10,  120, 250, 300]) # Example Infectious counts
R_data = np.array([0,   30,  150, 400]) # Example Recovered counts


# --- 2. DEFINE THE TRUE SIR MATH MODEL ---
def sir_model(y, t, beta, gamma):
    S, I, R = y
    # The standard SIR differential equations
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return [dSdt, dIdt, dRdt]


# --- 3. CREATE THE "GUESS CHECKER" (Objective Function) ---
# This function measures how wrong a guessed beta and gamma are compared to your real data
def check_guess(params):
    beta_guess, gamma_guess = params
    
    # Start the simulation using your actual Phase 1 numbers
    y0 = [S_data[0], I_data[0], R_data[0]] 
    
    # Run the SIR model using the guessed rates
    prediction = odeint(sir_model, y0, t_data, args=(beta_guess, gamma_guess))
    S_pred, I_pred, R_pred = prediction.T
    
    # Calculate the error (Sum of Squared Errors) between the prediction and your real survey data
    error = np.sum((S_data - S_pred)**2) + np.sum((I_data - I_pred)**2) + np.sum((R_data - R_pred)**2)
    return error


# --- 4. RUN THE OPTIMIZER TO FIND THE UNKNOWN RATES ---
# We give the computer a random starting guess: beta=0.5, gamma=0.1
initial_guess = [0.5, 0.1]

# The minimize function tests combinations until the "error" drops to the absolute minimum
# Bounds ensure beta and gamma don't become negative numbers
result = minimize(check_guess, initial_guess, bounds=[(0, 5), (0, 1)])

# Extract and print the final, calculated rates
best_beta, best_gamma = result.x

print("--- RESULTS ---")
print(f"Calculated Infection Rate (Beta):  {best_beta:.4f}")
print(f"Calculated Recovery Rate (Gamma): {best_gamma:.4f}")